# Notebook 09 — Shoreline Change QA in QGIS

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

cwd = Path.cwd()
if (cwd / "outputs").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "outputs").exists() and (cwd.parent / "notebooks").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve project root")

CHANGE_DIR = PROJECT_ROOT / "outputs" / "change"
RATES_PATH = CHANGE_DIR / "shoreline_change_rates.csv"
TRANSECTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "transects"
    / "2024"
    / "shoreline_transects_2024.gpkg"
)

rates = pd.read_csv(RATES_PATH)
transects = gpd.read_file(TRANSECTS_PATH)

valid_rates = pd.to_numeric(
    rates["change_rate_m_per_year"], errors="coerce"
).dropna()
print("Change-rate records:", len(rates))
print("Transects:", len(transects))
if valid_rates.empty:
    print("Rate range: no computed rates")
else:
    print(f"Rate range: {valid_rates.min():.3f} to {valid_rates.max():.3f} m/year")
print("Rate statuses:")
print(rates["status"].value_counts(dropna=False).to_string())

Change-rate records: 23
Transects: 23
Rate range: no computed rates
Rate statuses:
status
insufficient_data    23


In [2]:
transects_with_rates = transects.merge(
    rates,
    on="transect_id",
    how="left",
    validate="one_to_one",
)
OUTPUT_PATH = CHANGE_DIR / "shoreline_transects_with_rates.gpkg"
transects_with_rates.to_file(
    OUTPUT_PATH,
    driver="GPKG",
    layer="shoreline_transects_with_rates",
)
print("Saved:", OUTPUT_PATH)
print("Joined transects:", len(transects_with_rates))

Saved: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\outputs\change\shoreline_transects_with_rates.gpkg
Joined transects: 23


In [3]:
print(
    """QGIS shoreline-change inspection

1. Open the shoremonitor_qa project in QGIS.
2. Add outputs/change/shoreline_transects_with_rates.gpkg.
3. Style transects by change rate: red = erosion, grey = stable,
   blue = accretion, black = insufficient_data.
4. Check three high-erosion and three high-accretion transects.
5. For each, verify the yearly shoreline positions actually cross the
   transect near the coast, not along inland or NoData edges.
6. Record observations in
   outputs/qa/mombasa_test/2024/manual_shoreline_review_2024.csv
   using new review_ids starting at QA_CHANGE_001.
7. Do not edit geometries or change rates manually."""
)

QGIS shoreline-change inspection

1. Open the shoremonitor_qa project in QGIS.
2. Add outputs/change/shoreline_transects_with_rates.gpkg.
3. Style transects by change rate: red = erosion, grey = stable,
   blue = accretion, black = insufficient_data.
4. Check three high-erosion and three high-accretion transects.
5. For each, verify the yearly shoreline positions actually cross the
   transect near the coast, not along inland or NoData edges.
6. Record observations in
   outputs/qa/mombasa_test/2024/manual_shoreline_review_2024.csv
   using new review_ids starting at QA_CHANGE_001.
7. Do not edit geometries or change rates manually.


QA status remains **pending** until the QGIS inspection is completed. The coastal NoData issue remains unresolved.